# HDF5 files with Daft and h5py

This notebook uses the real DROID-style trajectory file in `examples/sample_data/trajectory.h5`. It covers the common usage spectrum: MIME detection, h5py-style object access, `Hdf5File` convenience methods, expression functions, and UDF patterns.

In [ ]:
from __future__ import annotations

from pathlib import Path

import daft
from daft.functions import guess_mime_type, hdf5_file, hdf5_keys, hdf5_read, hdf5_read_many, hdf5_visit

path = Path("sample_data/trajectory.h5")
trajectory = daft.Hdf5File(str(path))
df = daft.from_pydict({"path": [str(path)]}).select(hdf5_file(daft.col("path")).alias("trajectory"))

df.show()


In [ ]:
dataset_paths = {
    "joint_position": "action/joint_position",
    "gripper_position": "action/gripper_position",
}

## MIME detection

Daft can identify HDF5 content from the HDF5 signature bytes and returns the registered `application/vnd.hdfgroup.hdf5` media type.

In [ ]:
file_mime = daft.File(str(path)).mime_type()

bytes_df = daft.from_pydict({"data": [path.read_bytes()[:1024]]})
expr_mime = bytes_df.select(guess_mime_type(daft.col("data")).alias("mime")).collect().to_pydict()["mime"][0]

file_mime, expr_mime

## Python object usage

`Hdf5File` convenience methods mirror common h5py operations: hierarchical lookup, attributes, group traversal, metadata scans, and dataset reads. Remote files are materialized to a local temporary file before h5py opens them.

In [ ]:
root_keys = trajectory.keys(group="/")
datasets = [
    (obj["h5path"], obj["shape"], obj["dtype"])
    for obj in trajectory.metadata(group="/")
    if obj["kind"] == "dataset"
]
first_window = trajectory.read("action/joint_position")[:5]

root_keys, len(datasets), datasets[:12], first_window.shape

The convenience methods use HDF5-specific names: `dataset` for data arrays, `group` for group traversal, and `h5path` when either a group or dataset is valid. Use `metadata()` for a flattened metadata scan; `visit()` follows h5py traversal semantics and returns visited names when called without a callback.

In [ ]:
root_keys = trajectory.keys(group="/")
action_attrs = trajectory.attrs(h5path="action")
visited_names = trajectory.visit(group="/")
metadata_objects = trajectory.metadata(group="/")
dataset_objects = [obj for obj in metadata_objects if obj["kind"] == "dataset"]
arrays = trajectory.read(dataset_paths)

root_keys, action_attrs, visited_names[:10], dataset_objects[:5], arrays["joint_position"].shape, arrays["gripper_position"][:10]

## Expression functions

In a DataFrame, keep the HDF5 file as a lazy file reference. Use `hdf5_visit` for discovery, `hdf5_read` for one dataset, and `hdf5_read_many` when you need several datasets from the same file.

In [ ]:
df = daft.from_pydict({"path": [str(path)]}).select(
    hdf5_file(daft.col("path")).alias("trajectory")
)

expr_df = df.select(
    hdf5_keys(daft.col("trajectory"), group="/").alias("root_keys"),
    hdf5_visit(daft.col("trajectory"), group="/").alias("objects"),
    hdf5_read(daft.col("trajectory"), dataset="action/joint_position").alias("single_joint_position"),
    hdf5_read_many(daft.col("trajectory"), dataset_paths).alias("datasets"),
)

result = expr_df.collect().show()


`hdf5_read_many` emits a struct. Use `.unnest()` when you want each requested dataset as its own column.

In [ ]:
wide_df = df.select(hdf5_read_many(daft.col("trajectory"), dataset_paths).unnest())
wide = wide_df.collect().show()


## UDF usage

Use a UDF when the logic is not a simple dataset read: custom slicing, multiple operations on the same handle, attribute-aware branching, or third-party libraries that expect h5py objects.

In [ ]:
@daft.func(return_dtype=daft.DataType.float64())
def mean_joint_position(file: daft.Hdf5File) -> float:
    return float(file.read("action/joint_position").mean())


udf_result = df.select(mean_joint_position(daft.col("trajectory")).alias("mean_joint_position"))
udf_result.collect().show()

For very large remote HDF5 files, materializing once inside a UDF can be preferable when the logic will make many random reads. h5py then opens a real local file path instead of repeatedly calling back through a Python file-like object.

In [ ]:
@daft.func(return_dtype=daft.DataType.float64())
def mean_gripper_position_materialized(file: daft.Hdf5File) -> float:
    import h5py

    with file.to_tempfile() as tmp, h5py.File(tmp.name, "r") as h5:
        return float(h5["action/gripper_position"][:].mean())


materialized_result = df.select(
    mean_gripper_position_materialized(daft.col("trajectory")).alias("mean_gripper_position")
)
materialized_result.collect().to_pydict()

`Hdf5File` methods work on local files and Daft-supported URLs such as `hf://`, `s3://`, `gs://`, or configured protocol aliases. For custom h5py logic inside a UDF, materialize once with `file.to_tempfile()` and open the local path with `h5py.File(tmp.name, "r")`.